In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import lxml
from selenium import webdriver
import time


In [33]:
# %pip install lxml
# %pip install selenium

In [3]:
urls = {
    "US": "https://www.ysl.com/en-us/ca/shop-women/handbags",
    "FR": "https://www.ysl.com/en-fr/shop-women/handbags",
    "IT": "https://www.ysl.com/en-it/shop-women/handbags",
    "CN": "https://www.ysl.cn/categories/shop-women/handbags/view-all.html"
}

driver = webdriver.Chrome() # use selenium for javascript -- prices won't load with just soup

In [4]:
data = []

for country, url in urls.items():
    driver.get(url)
    time.sleep(5)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # china html is different
    if country == "CN":
        titles = soup.select(".productName")
        prices = soup.select(".productPrice")
    else:
        titles = soup.select('h2[data-qa^="plp-product-title"]')
        prices = soup.select('span[data-qa^="plp-product-price"]')

    # was for debugging
    # print("titles:", len(titles))
    # print("prices:", len(prices))

    for t,p in zip(titles, prices):
        name = t.get_text(strip=True)
        priceText = p.get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", priceText))

        if country == "CN": # china product ID is also stored differently - in the product URL only
            link = t.find_parent("a")["href"]
            sku = link.split("/")[-1].split("?")[0].replace(".html", "").upper()
        else:
            sku = t["data-qa"].replace("plp-product-title-", "")

        currency = re.findall(r"[^\d.,\s]", priceText)[0]

        data.append({
        "brand": "Saint Laurent",
        "product_name": name,
        "price": price,
        "currency": currency,
        "country": country,
        "sku": sku
})


In [5]:
df = pd.DataFrame(data)
print(df)



            brand               product_name    price currency country  \
0   Saint Laurent   MOMBASA small in leather   3450.0        $      US   
1   Saint Laurent   MOMBASA small in leather   3450.0        $      US   
2   Saint Laurent   MOMBASA small in leather   3450.0        $      US   
3   Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
4   Saint Laurent  MOMBASA medium in leather   4300.0        $      US   
..            ...                        ...      ...      ...     ...   
64  Saint Laurent            LOULOU中号绗缝小羊皮手袋  24500.0        ¥      CN   
65  Saint Laurent            LOULOU中号绗缝小羊皮手袋  24500.0        ¥      CN   
66  Saint Laurent            LOULOU小号绗缝小羊皮手袋  21500.0        ¥      CN   
67  Saint Laurent            LOULOU小号绗缝小羊皮手袋  21500.0        ¥      CN   
68  Saint Laurent            LOULOU小号绗缝小羊皮手袋  21500.0        ¥      CN   

                sku  
0   851432AAGWJ1000  
1   851432AAGWK2050  
2   851432AAGWK6195  
3   862029AAGWJ1000  
4

In [6]:
df.pivot(index="sku", columns="country", values="price")
# there are bunch of extra bags scraped from china + other mismatch but there are some we can compare across countries already (obv diff currencies rn)

country,CN,FR,IT,US
sku,,,,
801437AAEAX1000,21500.0,NaN,NaN,NaN
801437AAEAX1997,21500.0,NaN,NaN,NaN
801437AAEAX6197,21500.0,NaN,NaN,NaN
801439AAEAX1000,24500.0,NaN,NaN,NaN
801439AAEAX1997,24500.0,NaN,NaN,NaN
801439AAEAX6197,24500.0,NaN,NaN,NaN
8098241U8P72916,29900.0,NaN,NaN,NaN
8098241U8P73194,29900.0,NaN,NaN,NaN
809824AAB321000,29900.0,NaN,NaN,NaN


In [7]:
df.to_csv("data/ysl_bags_prices_preliminary.csv", index=False)

In [43]:
# TODO:

# it's repeating bags that are the same style + price, but just the color is diff
# figure out how to scroll to the bottom and scrape all bags
# get product URL?
# convert everything to USD

In [49]:
# print(soup.prettify()[:1000000])